In [ ]:
# -*- coding: utf-8 -*-
import json
import pandas as pd
from pathlib import Path

output_root = Path('lstm_multivariate_outputs')
latest_info = json.loads((output_root / 'latest.json').read_text(encoding='utf-8'))
run_dir = output_root / latest_info['output_directory']
as_of_date = latest_info['as_of_date']

next_day = pd.read_csv(run_dir / 'next_day_recommendation.csv')
review_path = run_dir / 'daily_portfolio_review.csv'
portfolio_review = pd.read_csv(review_path) if review_path.exists() else pd.DataFrame()

log_lines = [f"## 📅 {as_of_date} AI 每日模擬交易與動態權重檢討日誌 (Run: `{latest_info['run_id']}`)\n"]

# 1. 檢討前日模擬交易損益
if not portfolio_review.empty:
    last_rev = portfolio_review.iloc[-1]
    pred_ret = last_rev['Predicted_Gross_Return_Pct']
    act_ret = last_rev['Actual_Gross_Return_Pct']
    cash_w = last_rev['Cash_Weight'] * 100
    status_icon = "✅ 方向預測正確" if last_rev['Gross_Direction_Correct'] else "⚠️ 市場波動偏離預期"
    log_lines.append(
        f"### 🔍 前次模擬交易收盤對答案 ({last_rev['Target_Date']})\n"
        f"- **AI 預期投組毛報酬**：`{pred_ret:+.2f}%` ｜ **真實結算毛報酬**：`{act_ret:+.2f}%` ({status_icon})\n"
        f"- **當日現金防禦部位**：`{cash_w:.1f}%`\n"
    )
else:
    log_lines.append("### 🔍 前次模擬交易收盤對答案\n- 今日完成首次模型部署與明日建倉建議，自下一交易日收盤起自動進行跨日損益核對。\n")

# 2. 明日動態權重決策說明
risk_off = bool(next_day['Risk_Off'].iloc[0])
vol_20d = next_day['Market_Vol_20d'].iloc[0]
vol_th = next_day['Risk_Threshold_Pct'].iloc[0]
target_date = next_day['Target_Date'].iloc[0]

risk_msg = (
    f"🚨 **觸發高波動避險濾網** (大盤20日波動率 `{vol_20d:.2f}%` > 風控線 `{vol_th:.2f}%`)，總持股上限強制縮減至 50%！"
    if risk_off else
    f"🟢 **市場波動正常** (大盤20日波動率 `{vol_20d:.2f}%` <= 風控線 `{vol_th:.2f}%`)，允許最高 100% 資金配置。"
)

log_lines.append(f"### 🎯 下一交易日 ({target_date}) 模擬交易下單與動態權重調整\n- **系統風控判讀**：{risk_msg}\n")
for _, row in next_day.iterrows():
    w_pct = row['Target_Weight'] * 100
    sent = row['Daily_Sentiment']
    pred = row['Predicted_Return_Pct']
    if w_pct >= 25:
        reason = "🟢 **預測看漲且新聞情緒樂觀 (>0.3)，動態加碼至上限**"
    elif w_pct == 0:
        reason = "🔴 **預測看跌或新聞情緒悲觀 (<-0.3)，清倉避險 (0%)**"
    else:
        reason = "⚪ **趨勢與情緒中性，維持基本權重配置**"
    log_lines.append(
        f"  - **{row['Ticker']}**：新聞情緒 `{sent:+.2f}` ｜ 預測明日報酬 `{pred:+.2f}%` ➔ **目標權重 `{w_pct:.1f}%`** ({reason})"
    )

log_lines.append(f"- **💵 保留現金部位**：**`{next_day['Cash_Weight'].iloc[0]*100:.1f}%`**\n\n---\n\n")

journal_path = Path('DAILY_TRADING_JOURNAL.md')
old_content = journal_path.read_text(encoding='utf-8') if journal_path.exists() else "# 🤖 多變量 LSTM 每日模擬交易與動態權重檢討日誌\n\n"
header, _, body = old_content.partition("\n\n")
journal_path.write_text(header + "\n\n" + "\n".join(log_lines) + body, encoding='utf-8')
print("✅ 已更新 DAILY_TRADING_JOURNAL.md！")

FileNotFoundError: [Errno 2] No such file or directory: 'lstm_multivariate_outputs/latest.json'